# Developing with ChatGPT (new OpenAI Responses API)

This notebook explores different ways to use GPT models as simple assistants.

**Hard Constraints**

- One prompt
- One model call
- No memory
- No retrieval

**Main Scenarios**

- Summarizing text
- Detecting sentiment in a text
- Classifying mental disorders
- Detecting language
- Translation
- Changing tone
- Troubleshooting software problems from an error message

## Setup

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
client = OpenAI()
client.api_key = os.environ["OPENAI_API_KEY"]

In [2]:
# Get completion for a single prompt (str) or a chat (list[dict])
def get_completion(
    prompt,
    model="gpt-4o-mini",
    temperature=0,
    max_output_tokens=100
):
    response = client.responses.create(
        input=prompt,
        model=model,
        max_output_tokens=max_output_tokens,
        temperature=temperature
    )

    return response.output[0].content[0].text

## Text summarization

In [3]:
DEF_GPT_MODEL = "gpt-4o-mini"
DELIMITER = "````"
MAX_WORDS = 100
MAX_CHARS_IN = 20_000

In [4]:
SUMMARY_PROMPT = """\
Your task is to summarize a text into bullet point items. The input text will be delimited by {f_delimiter}. \
Limit your response to {f_max_words} words and don't output incomplete sections and sentences. \
Text to summarize : {f_delimiter}{f_text}{f_delimiter}"""

In [5]:
def summarize_text(
    text: str,
    delimiter=DELIMITER,
    max_words=MAX_WORDS,
    max_chars_in = MAX_CHARS_IN,
    model: str=DEF_GPT_MODEL,
    debug=False) -> str:
    """ Uses given GPT model to summarize input text into bullet list format.
    If input text is longer than expected, it will be truncated to configured
    maximum value. """

    if not text:
        raise ValueError("The value of 'text' parameter cannot be None or an empty string.")

    text = text.replace("\n", " ").strip()
    if len(text) > max_chars_in:
        text = text[:max_chars_in]

    max_tokens = int(max_words * 4. / 3)  # simple approximation
    prompt = SUMMARY_PROMPT.format(
        f_delimiter=delimiter, f_max_words=max_words, f_text=text)
    if debug:
        print(f"Input prompt :\n{prompt}")
    response = get_completion(prompt, model=model, max_output_tokens=max_tokens)

    return response

In [6]:
def summarize_file(
    path,
    delimiter=DELIMITER,
    max_words=MAX_WORDS,
    max_chars_in = MAX_CHARS_IN,
    model: str=DEF_GPT_MODEL) -> str:
    """ Uses given GPT model to summarize contents of given text file
    into bullet list format. If text content is longer than expected,
    the rest of file contents will be truncated. Raises error if given
    file is binary, invalid or inaccessible. """

    try:
        with open(path, "r") as file:
            text = file.read()

        summary = summarize_text(
            text, delimiter=delimiter, max_words=max_words,
            max_chars_in=max_chars_in, model=model)
        return summary
    except Exception as ex:
        print(f"[ERROR] {ex}")

In [7]:
SUMMARY_PROMPT.format(
    f_delimiter=DELIMITER, f_max_words=MAX_WORDS, f_text="Some text")

"Your task is to summarize a text into bullet point items. The input text will be delimited by ````. Limit your response to 100 words and don't output incomplete sections and sentences. Text to summarize : ````Some text````"

In [8]:
sample_text = f"""
What is Agentic AI?

Agentic AI is an autonomous AI system that can act independently to achieve pre-determined goals. Traditional software follows pre-defined rules, and traditional artificial intelligence also requires prompting and step-by-step guidance. However, agentic AI is proactive and can perform complex tasks without constant human oversight. "Agentic" indicates agency — the ability of these systems to act independently, but in a goal-driven manner.

AI agents can communicate with each other and other software systems to automate existing business processes. But beyond static automation, they make independent contextual decisions. They learn from their environment and adapt to changing conditions, enabling them to perform sophisticated workflows with accuracy.

For example, an agentic AI system can optimize employee shift schedules. If an employee is off sick, the agent can communicate with other employees and readjust the schedule while still meeting project resource and time requirements.
What are the characteristics of agentic AI systems?

Here are the key features of an agentic AI system.
Proactive

Agentic AI acts proactively rather than waiting for direct input. Traditional systems are reactive, responding only when triggered and following predefined workflows. In contrast, agentic systems anticipate needs, identify emerging patterns, and take initiative to address potential issues before they escalate. Their proactive behavior is driven by environmental awareness and their ability to evaluate outcomes against long-term goals.

For instance, in a supply chain setting, a traditional logistics platform updates delivery statuses when a user checks in or through periodic notifications. An agentic AI system, however, can monitor inventory levels, track weather conditions, and anticipate shipping delays. It can proactively raise alerts and even reroute shipments to reduce downtime.
Adaptable

A key feature of agentic AI is its ability to adapt to changing environments and specific domains. Traditional SaaS solutions are built to scale across industries and handle repetitive tasks, but they often lack the depth to understand unique domain-specific situations. Agentic systems fill this gap by using context awareness and domain knowledge, enabling AI agents to respond intelligently. They adjust their actions based on real-time input and can handle complex scenarios that standard solutions cannot.

For example, while a generic customer service platform might respond with predefined answers, an agentic AI system supporting a healthcare provider understands medical terminology and complies with healthcare regulations. It can adapt to evolving patient concerns and delivers more accurate, context-sensitive support.
Collaborative

Agentic AI is designed to collaborate with humans and with other agentic AI systems. AI agents work as part of a broader team. They can understand shared goals, interpret human intent, and coordinate actions accordingly. They work well in settings that require human oversight or decision-making by considering inputs from multiple sources.

For example, a treatment planning agent can coordinate with several different medical teams to prepare an integrated treatment and follow-up plan for a cancer patient.
Specialized

Agentic AI typically builds upon multiple hyperspecialized agents, with each focused on a narrow area of expertise. These AI-powered agents coordinate with each other, sharing insights and handing off tasks as needed. This approach enables significantly deeper domain-specific performance.

For instance, in financial services, one agent might specialize in regulatory compliance, another in fraud detection, and another in portfolio optimization. Working together, they can monitor transactions in real time, flagging anomalies and recommending investment adjustments, all while maintaining regulatory compliance.
What are the use cases of agentic AI?

Agentic AI has unlimited applications and can be fully customized to any requirement. We give some examples of early adoption.
Supporting research and development

Research and development in any field requires a great deal of manual processes, such as testing hypotheses, gathering research information, collecting data, synthesizing insights across data sources, and more. Agentic AI can reduce the need for human intervention with these manual processes. It streamlines research and better coordinates teams that are working on research and development challenges.

Agentic AI also facilitates multi-agent orchestration, where supervisors use multiple specialist models to construct complex research and development pipelines. For example, agentic AI could draw from recent research published on credible platforms, synthesize the results, plan further tests, and present researchers with the final product they need to investigate. This approach saves a significant amount of time and cost involved in research.
Code transformation

Agentc AI can use specialized AI-powered agents to remove the complexity of modernization and migration tasks. For example, agentic AI models for .NET can modernize Windows-based .NET applications to Linux significantly faster using machine learning, graph neural networks, Large language models (LLMs), and automated reasoning.

Equally, agentic AI can decompose monolithic z/OS COBOL applications into individual components, reducing the timeframe of this process from months to minutes. Agentic AI offers unmatched speed, scale, and performance in automating application migration and modernization.
Incident response automation

Whenever an incident occurs, whether due to a vulnerability or a manual error, agentic AI can expedite the incident response process, saving your business time and improving time-to-recovery. Agentic AI can automate the entire incident response pathway, rolling back issues, creating incident reports, and notifying any team members who need to stay informed.

Agentic AI enhances incident response speed while also providing a more specific and in-depth post-incident analysis to prevent the same errors from recurring in the future.
Customer service automation

In many customer service scenarios, the information that a customer needs is already online in a tutorial or help article. Agentic AI processes customer service inquiries and rapidly searches through available company documents to find a suitable answer that helps them out. If this alone isn’t enough to solve a query, agentic AI can then communicate with the user to gather more information about their case and direct them toward a solution. They are designed with modular components, such as reasoning engines, memory, cognitive skills, and tools, that enable them to remedy the vast majority of problems.

AI-powered agents can operate independently, learn from their environment, adapt to changing conditions, and develop more effective strategies to assist customers. If, after several attempts, it cannot solve a customer’s issue, it then contacts a human support agent and assigns them to the case. Utilizing this form of AI in customer service scenarios alleviates the burden on human teams and enables the vast majority of customer-oriented services to operate 24/7.
What are the benefits of agentic AI?

There are several business benefits to using agentic AI.
Increased efficiency

Agentic artificial intelligence enables businesses to simplify the complexity of various challenging or specialized tasks through automation. Instead of relying on human-driven manual practices, using agentic AI can automate tedious processes, freeing up time for your employees. Your employees can use the extra time that agentic AI saves them on more demanding tasks, such as problem-solving, strategic planning, and other drivers of growth.
Increased user trust

Agentic AI can offer a higher degree of personalization when interacting with customers. By utilizing existing customer data, agentic AI can quickly produce tailored messaging, engage with the customer in their preferred tone, and offer practical product recommendations. Over time, agentic AI improves customer relationships and builds trust between customers and your business.

Businesses can also utilize agentic artificial intelligence to analyze customer feedback, identify the most frequently occurring information, and provide it to product engineers. It can also directly respond to users who leave feedback, creating positive feedback loops where customers feel that their feedback is taken seriously by your company.
Continuous improvement

Agentic AI can continuously learn and improve, adapting to any tasks assigned to it. It interacts, learns from feedback, and optimizes its decision-making based on this recursive loop. For businesses, this means that it continues to deliver its benefits at higher and higher levels over time.
Human augmentation

Agentic AI can serve as a fantastic collaboration tool for human agents, enhancing their productivity and reducing the number of laborious manual tasks they must complete. By working alongside agentic AI models, human agents can overcome complex challenges, automate difficult decision-making pathways, and drive their efficiency.
What are the types of agentic AI systems?

Agentic AI can be single or multi-agent setups. In a single-agentic AI system, one AI agent handles all tasks sequentially. These are preferable when businesses need a faster solution that can work on a well-defined problem or process.

Multi-agentic AI, on the other hand, involves multiple AI agents collaborating to break down complex workflows into smaller segments. This approach is more scalable than single systems and is much more flexible for solving complex scenarios. The vast majority of agentic AI agents refer to this latter, more diverse form of AI deployment.

Here are a few different structures of multi-agent systems.
Horizontal multi-agent

Horizontal multi-agent AI is a system of working where every AI agent has the same level of technical proficiency and complexity. Each agent specializes in a narrow skill, bringing their findings together to solve a complex problem. This structure utilizes lateral collaboration and communication among the specialized AI agents.
Vertical multi-agent

In a vertical multi-agent system, there is a hierarchical structure in which lower-level AI agents have ‘easier’ tasks compared to the higher ones. The highest levels of this structure handle tasks that require more processing power and LLMs, such as critical thinking, reasoning, and decision-making. Lower-level AI agents in this structure perform tasks such as collecting data, formatting it, or processing it to pass it to higher levels.
"""

In [9]:
summary = summarize_text(sample_text, max_words=300)
word_count = len(summary.split(" "))
print(f"\nSummary :\n{summary}")
print(f"\nSummary text has {word_count} words.")


Summary :
- **Definition of Agentic AI**: 
  - Autonomous AI system acting independently to achieve goals.
  - Proactive, capable of complex tasks without constant human oversight.

- **Key Characteristics**:
  - **Proactive**: Anticipates needs and addresses issues before they escalate.
  - **Adaptable**: Adjusts actions based on real-time input and context awareness.
  - **Collaborative**: Works with humans and other AI systems, understanding shared goals.
  - **Specialized**: Composed of multiple agents focused on specific areas of expertise.

- **Use Cases**:
  - **Research and Development**: Streamlines manual processes, coordinates teams, and synthesizes insights.
  - **Code Transformation**: Automates modernization and migration tasks, significantly reducing time.
  - **Incident Response Automation**: Expedites response processes and improves recovery times.
  - **Customer Service Automation**: Processes inquiries, finds solutions, and escalates unresolved issues to human agent

In [10]:
sample_file = "../src/genai_lab/samples/agentic-ai-aws.txt" # Same content
summary = summarize_file(sample_file, max_words=300)
word_count = len(summary.split(" "))
print(f"\nSummary :\n{summary}")
print(f"\nSummary text has {word_count} words.")


Summary :
- **Definition of Agentic AI**: 
  - Autonomous AI system acting independently to achieve goals.
  - Proactive, capable of complex tasks without constant human oversight.

- **Key Characteristics**:
  - **Proactive**: Anticipates needs and addresses issues before they escalate.
  - **Adaptable**: Adjusts actions based on real-time input and domain-specific knowledge.
  - **Collaborative**: Works with humans and other AI systems, understanding shared goals.
  - **Specialized**: Composed of multiple agents focused on narrow areas of expertise.

- **Use Cases**:
  - **Research and Development**: Streamlines manual processes, coordinates teams, and synthesizes insights.
  - **Code Transformation**: Automates modernization and migration tasks, significantly reducing time.
  - **Incident Response Automation**: Expedites incident response, creating reports and notifying teams.
  - **Customer Service Automation**: Processes inquiries, finds solutions, and escalates unresolved issues

## Sentiment detection